# 02b · Landsat views of the unsure validation points

**Project:** Paddy Burnt-Area Intelligence (Satellite Mission Intelligence for India)
**Question for this notebook:** do Landsat 8 and 9 add clear views where Sentinel-2 had none, so that validation points labelled `unsure` can be settled?

**Why:** 56 of the 240 validation points in Notebook 02 are `unsure`. Many are `gap` cases: Sentinel-2 had no clear view for a week or more while the straw went, most often from 25 Oct to 2 Nov. Landsat 8 and 9 pass on other days, including a nearly cloud-free 27 Oct. Settling some of these points narrows the range Notebook 02 reports for the burnt area, and gives Notebook 03 more labelled points.

**How it works**
1. NASA's Harmonized Landsat Sentinel-2 product (HLS, version 2.0) supplies Landsat 8/9 surface reflectance at 30 m on the same grid as Sentinel-2 (product HLSL30). It is free, with an Earthdata login.
2. The notebook finds the HLSL30 scenes over the district between `HLS_START` and `HLS_END`. It reads only small windows around the points (630 m across), never whole scenes.
3. Each point gets an image strip of the dates on which its 60 m block was clear, in true colour and in false colour, like the Sentinel-2 strips. These go in `data/raw/validation_chips/P001_landsat.png`.
4. A list says which Landsat dates were clear at each point. A table of the scenes, with their cloud cover, goes to the mission trade-off.

**Before you run it:** put `EARTHDATA_TOKEN=your_token` in the `.env` file in the project folder. To make a token, sign in at urs.earthdata.nasa.gov and open *Generate Token*. Alternatively, put `EARTHDATA_USERNAME=…` and `EARTHDATA_PASSWORD=…`. In Colab, add the same names as Secrets.

**Run time:** the first run reads a few thousand small windows, which takes roughly 10 to 20 minutes on a home connection. Reruns read nothing: the windows are cached in `data/raw/hls/`, which git ignores. No credits are used.

**What it produces**

| Output | Where |
|---|---|
| Landsat strip for each point | `data/raw/validation_chips/P001_landsat.png` |
| Clear Landsat dates at each point, for relabelling | `data/processed/validation_landsat_views_sangrur_2025.csv` |
| HLSL30 scenes over the district, with cloud cover | `data/processed/hls_l30_scenes_sangrur_2025.csv` |
| Source register row | `sources/source_register.csv` |

In [ ]:
# Colab: add libraries its default image may lack. On your own computer, install requirements.txt into the
# project's environment first; there this cell only checks that the install worked.
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401  (this module exists only on Colab)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

NEEDED = ('earthaccess', 'rasterio', 'pyproj', 'shapely', 'matplotlib', 'pandas', 'requests')
if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'earthaccess', 'rasterio'], check=True)
else:
    missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError('Missing: ' + ', '.join(missing) + ". Install requirements.txt into the "
                                  "project's environment and re-run.")
    print('Libraries found.')

## 1. Settings

In [ ]:
import os, json, math, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import rasterio
from rasterio.windows import Window
from pyproj import Transformer
from shapely.geometry import Point, Polygon
from IPython.display import display
import earthaccess

TODAY = pd.Timestamp.today().normalize()
ACCESS_DATE = TODAY.strftime('%Y-%m-%d')

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/India-Space-Mission-Intelligence')
elif os.environ.get('PROJECT_DIR'):
    PROJECT = Path(os.environ['PROJECT_DIR']).resolve()
else:
    # On your computer: walk up from the working folder to the project root (the folder with requirements.txt).
    here = Path.cwd().resolve()
    PROJECT = next((p for p in (here, *here.parents)
                    if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), here.parent)

PROCESSED = PROJECT / 'data' / 'processed'
CHIPS = PROJECT / 'data' / 'raw' / 'validation_chips'
SOURCES = PROJECT / 'sources'
print('Project folder:', PROJECT)

# --- What to look at ---------------------------------------------------------------------------------
SUFFIX = 'sangrur_2025'                 # Notebook 02's whole-district outputs
HLS_START, HLS_END = '2025-10-10', '2025-11-25'   # Landsat dates to look at (both included)
POINT_SET = 'unsure'                    # 'unsure': the points to relabel; 'all': every validation point
MAX_SCENE_CLOUD = 90                    # skip scenes whose cloud cover, over the whole 110 km tile, is above this (%)
SENTINEL2_GAP = ('2025-10-26', '2025-11-01')   # days with no clear Sentinel-2 view over most `gap` points

# --- Reading -----------------------------------------------------------------------------------------
PANEL_PX = 21                           # window around each point: 21 Landsat pixels of 30 m = 630 m
BANDS = {'blue': 'B02', 'green': 'B03', 'red': 'B04', 'nir': 'B05', 'swir2': 'B07'}   # HLSL30 band files
SCALE = 0.0001                          # HLS reflectance is stored as integers x 10,000; -9999 = no data
WORKERS = 6                             # scenes read at the same time
CACHE = PROJECT / 'data' / 'raw' / 'hls' / SUFFIX   # windows already read (git ignores data/raw)
GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif',
                GDAL_HTTP_MULTIRANGE='YES', GDAL_HTTP_MERGE_CONSECUTIVE_RANGES='YES',
                GDAL_HTTP_MAX_RETRY='4', GDAL_HTTP_RETRY_DELAY='2')


def read_env_file(path):
    # KEY=value lines from a .env file; blank lines and lines starting with # are skipped.
    values = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8-sig').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                name, value = line.split('=', 1)
                values[name.strip()] = value.strip().strip('"\'')
    return values


def earthdata_login():
    # Puts the Earthdata token (or username and password) where earthaccess looks for them, without printing them.
    from_file = read_env_file(PROJECT / '.env')
    for name in ('EARTHDATA_TOKEN', 'EARTHDATA_USERNAME', 'EARTHDATA_PASSWORD'):
        value = None
        if IN_COLAB:
            try:
                value = userdata.get(name)
            except Exception:
                value = None
        value = value or os.environ.get(name) or from_file.get(name)
        if value:
            os.environ[name] = value.strip()
    if not (os.environ.get('EARTHDATA_TOKEN')
            or (os.environ.get('EARTHDATA_USERNAME') and os.environ.get('EARTHDATA_PASSWORD'))):
        raise RuntimeError('No Earthdata login found. Put the line EARTHDATA_TOKEN=your_token in the .env file in '
                           f'{PROJECT} (make a token at urs.earthdata.nasa.gov, under Generate Token), then re-run.')
    auth = earthaccess.login(strategy='environment')
    if not (auth and getattr(auth, 'authenticated', False)):
        raise RuntimeError('Earthdata login failed. Tokens expire: make a new one at urs.earthdata.nasa.gov and '
                           'put it in .env.')
    return auth


AUTH = earthdata_login()
print('Earthdata login worked (nothing from .env is printed).')

## 2. Helper functions

How a window is read: the download address of each image file is first exchanged for a signed link, through one redirect to NASA's distribution server. GDAL then reads just the bytes it needs from that link. In a test on 31 Oct 2026, opening one image file took about 5 seconds, most of it getting the signed link, and each window then took under a second. The library's default way of reading took about 30 seconds per window.

In [ ]:
REGISTER = SOURCES / 'source_register.csv'
REGISTER_COLUMNS = ['source_id', 'dataset_or_product', 'provider', 'url', 'date_accessed', 'geography',
                    'time_range', 'resolution', 'licence_or_terms', 'preprocessing', 'known_limitations', 'used_in']
NOTEBOOK = 'notebooks/02b_landsat_views.ipynb'


def register_source(**row):
    # Add or replace one row of the source register, keyed by source_id.
    row = {c: str(row.get(c, '')) for c in REGISTER_COLUMNS}
    reg = pd.read_csv(REGISTER, dtype=str).fillna('') if REGISTER.exists() else pd.DataFrame(columns=REGISTER_COLUMNS)
    reg = reg[reg['source_id'] != row['source_id']]
    pd.concat([reg, pd.DataFrame([row])], ignore_index=True)[REGISTER_COLUMNS].to_csv(REGISTER, index=False)


def scene_info(granule):
    # Name, MGRS tile, date and cloud cover of one HLSL30 scene, plus its outline (lon/lat) if CMR gives one.
    umm = granule['umm']
    name = umm['GranuleUR']                              # e.g. HLS.L30.T43REQ.2025300T053042.v2.0
    parts = name.split('.')
    tile, stamp = parts[2], parts[3]
    date = pd.Timestamp(year=int(stamp[:4]), month=1, day=1) + pd.Timedelta(int(stamp[4:7]) - 1, unit='D')
    cloud = next((float(a['Values'][0]) for a in umm.get('AdditionalAttributes', [])
                  if a.get('Name') == 'CLOUD_COVERAGE'), np.nan)
    outline = None
    try:
        geometry = umm['SpatialExtent']['HorizontalSpatialDomain']['Geometry']
        boundary = geometry['GPolygons'][0]['Boundary']['Points']
        outline = Polygon([(p['Longitude'], p['Latitude']) for p in boundary])
    except (KeyError, IndexError, TypeError):
        pass
    links = {u.rsplit('.', 2)[-2]: u for u in granule.data_links() if u.endswith('.tif')}
    return {'scene': name, 'tile': tile, 'date': date, 'cloud_cover_pct': cloud, 'outline': outline,
            'links': links}


_local = threading.local()


def session():
    # One logged-in requests session per thread.
    if not hasattr(_local, 'session'):
        _local.session = earthaccess.get_requests_https_session()
    return _local.session


def signed_link(url, tries=3):
    # Follow the redirect from NASA's address to the signed link GDAL can read directly.
    last = ''
    for attempt in range(tries):
        try:
            response = session().get(url, stream=True, timeout=120)
            final, status = response.url, response.status_code
            response.close()
            if status == 200:
                return final
            last = f'HTTP {status}'
        except requests.RequestException as err:
            last = type(err).__name__
        time.sleep(2 * (attempt + 1))
    raise RuntimeError(f'Could not get a download link for {url.rsplit("/", 1)[-1]} ({last}).')


@lru_cache(maxsize=None)
def to_crs(crs_wkt):
    return Transformer.from_crs('EPSG:4326', crs_wkt, always_xy=True)


def window_at(src, lon, lat):
    # The PANEL_PX x PANEL_PX window centred on a point, and the point's position inside it (fractional pixels),
    # or None if the window doesn't fit inside this scene.
    x, y = to_crs(src.crs.to_wkt()).transform(lon, lat)
    row, col = src.index(x, y, op=float)
    r0, c0 = int(math.floor(row)) - PANEL_PX // 2, int(math.floor(col)) - PANEL_PX // 2
    if r0 < 0 or c0 < 0 or r0 + PANEL_PX > src.height or c0 + PANEL_PX > src.width:
        return None
    return Window(c0, r0, PANEL_PX, PANEL_PX), (row - r0, col - c0)


def block_mask(pos):
    # Pixels whose centres lie within 30 m (one pixel) of the point: the point's 60 m block.
    rows = np.arange(PANEL_PX)[:, None] + 0.5
    cols = np.arange(PANEL_PX)[None, :] + 0.5
    return (np.abs(rows - pos[0]) <= 1) & (np.abs(cols - pos[1]) <= 1)


def judge_block(fmask, pos):
    # Clear when most of the block has no cloud, cloud shadow or cloud edge (HLS Fmask bits 1, 3 and 2);
    # hazy when most of it carries the 'high aerosol' level (bits 6-7 = 3). 255 = no data.
    block = fmask[block_mask(pos)].astype(int)
    valid = block != 255
    clear = valid & ((block & 0b1110) == 0)
    hazy = valid & (((block >> 6) & 0b11) == 3)
    return bool(clear.mean() >= 0.5), bool(hazy.mean() >= 0.5)


def read_scene(info, points):
    # Read the Fmask window around every point inside the scene, then the five bands where the block is clear.
    # Returns {point_id: {...}} for the points inside, and the number of image files opened.
    found, opened = {}, 0
    with rasterio.Env(**GDAL_ENV):
        with rasterio.open(signed_link(info['links']['Fmask'])) as src:
            opened += 1
            for pid, lon, lat in points[['point_id', 'lon', 'lat']].itertuples(index=False):
                placed = window_at(src, lon, lat)
                if placed is None:
                    continue
                window, pos = placed
                fmask = src.read(1, window=window)
                clear, hazy = judge_block(fmask, pos)
                found[pid] = {'pos': pos, 'window': window, 'fmask': fmask, 'clear': clear, 'hazy': hazy}
        wanted = [pid for pid, v in found.items() if v['clear']]
        if wanted:
            for band in BANDS.values():
                with rasterio.open(signed_link(info['links'][band])) as src:
                    opened += 1
                    for pid in wanted:
                        found[pid][band] = src.read(1, window=found[pid]['window'])
    for v in found.values():
        v.pop('window')
    return found, opened


def cache_path(scene):
    return CACHE / f'{scene}.npz'


def save_scene(scene, found, checked):
    arrays = {'checked': np.array(sorted(checked))}
    meta = {}
    for pid, v in found.items():
        meta[pid] = {'pos': list(v['pos']), 'clear': v['clear'], 'hazy': v['hazy']}
        arrays[f'{pid}__fmask'] = v['fmask']
        for band in BANDS.values():
            if band in v:
                arrays[f'{pid}__{band}'] = v[band]
    arrays['meta'] = np.array(json.dumps(meta))
    CACHE.mkdir(parents=True, exist_ok=True)
    partial = CACHE / f'{scene}.partial.npz'            # written first, so an interrupted run leaves no bad file
    np.savez_compressed(partial, **arrays)
    os.replace(partial, cache_path(scene))


def load_scene(scene):
    # Cached windows of one scene: (found, checked point IDs), or None if not cached.
    path = cache_path(scene)
    if not path.exists():
        return None
    with np.load(path) as data:
        meta = json.loads(str(data['meta']))
        found = {}
        for pid, m in meta.items():
            v = {'pos': tuple(m['pos']), 'clear': m['clear'], 'hazy': m['hazy'], 'fmask': data[f'{pid}__fmask']}
            for band in BANDS.values():
                if f'{pid}__{band}' in data:
                    v[band] = data[f'{pid}__{band}']
            found[pid] = v
        return found, set(data['checked'].tolist())


def reflectance(raw):
    values = raw.astype('float32')
    values[raw == -9999] = np.nan
    return values * SCALE


def as_image(rgb):
    # Clip to 0-1 and show a pixel grey when any of its three bands is missing.
    rgb = np.clip(rgb, 0, 1)
    rgb[~np.isfinite(rgb).all(axis=-1)] = 0.85
    return rgb


def landsat_strip(pid, views, out_path, ncols=12):
    # The point's clear Landsat dates: true colour above SWIR false colour (the same stretches as the
    # Sentinel-2 strips), 630 m across, with a square just outside the 60 m block. Returns the dates shown.
    ncols = min(ncols, max(len(views), 1))
    pairs = max(-(-len(views) // ncols), 1)
    heights = ([1, 1, 0.35] * pairs)[:-1]
    fig, grid = plt.subplots(len(heights), ncols, figsize=(ncols * 1.25, pairs * 2.9 + 0.6), squeeze=False,
                             gridspec_kw={'height_ratios': heights, 'hspace': 0.08, 'wspace': 0.06})
    for ax in grid.ravel():
        ax.set_axis_off()
    axes = np.vstack([grid[3 * p: 3 * p + 2] for p in range(pairs)])
    for i, (date, v) in enumerate(views):
        pair, col = divmod(i, ncols)
        refl = {name: reflectance(v[band]) for name, band in BANDS.items()}
        images = (as_image(np.stack([refl['red'], refl['green'], refl['blue']], axis=-1) * 3.0),
                  as_image(np.stack([refl['swir2'] / 0.35, refl['nir'] / 0.35, refl['red'] / 0.20], axis=-1)))
        # Metres from the point: the window spans PANEL_PX pixels of 30 m; the point sits at v['pos'].
        r, c = v['pos']
        ext = (-c * 30, (PANEL_PX - c) * 30, -(PANEL_PX - r) * 30, r * 30)
        for ax, img in zip((axes[2 * pair, col], axes[2 * pair + 1, col]), images):
            ax.imshow(img, extent=ext, interpolation='nearest')
            ax.add_patch(Rectangle((-45, -45), 90, 90, fill=False, edgecolor='cyan', linewidth=1.2))
            ax.set_xlim(ext[0], ext[1])
            ax.set_ylim(ext[2], ext[3])
        axes[2 * pair, col].set_title(f'{date:%d %b}' + (' (haze)' if v['hazy'] else ''), fontsize=8)
    if not views:
        axes[0, 0].text(0.5, 0.5, 'No clear Landsat date in this period.', ha='center', va='center')
    for pair in range(pairs):
        for r, name in ((2 * pair, 'true colour'), (2 * pair + 1, 'SWIR false colour')):
            axes[r, 0].text(-0.08, 0.5, name, rotation=90, transform=axes[r, 0].transAxes, ha='right', va='center',
                            fontsize=7)
    fig.suptitle(f'{pid}: Landsat 8/9 at 30 m (HLS). Was the field in the square burnt on any date? '
                 '(only dates with a clear view are shown)\nBurns: black or dark grey in true colour, dark maroon to '
                 'black in false colour. Soil: brown / pink. Straw: pale / white. Damp soil: dark brown / dark '
                 'blue-purple. Pixels are 3x coarser than the Sentinel-2 dates.', fontsize=8)
    fig.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    return len(views)

## 3. Points to look at

The validation points and their labels come from Notebook 02. With `POINT_SET = 'unsure'`, only the `unsure` points are read. The map's class for each point isn't shown here, so relabelling stays blind.

In [ ]:
points_path = PROCESSED / f'validation_points_{SUFFIX}.csv'
labels_path = PROCESSED / f'validation_labels_{SUFFIX}.csv'
for path in (points_path, labels_path):
    if not path.exists():
        raise RuntimeError(f'{path} is missing: run Notebook 02 for the whole district first.')
labels = pd.read_csv(labels_path, dtype=str).fillna('')
labels['answer'] = labels['burnt'].str.strip().str.lower()
pts = pd.read_csv(points_path)[['point_id', 'lon', 'lat']].merge(labels[['point_id', 'answer', 'notes']],
                                                                  on='point_id')
if POINT_SET == 'unsure':
    pts = pts[pts['answer'] == 'unsure'].reset_index(drop=True)
elif POINT_SET != 'all':
    raise ValueError("POINT_SET must be 'unsure' or 'all'.")
if pts.empty:
    raise RuntimeError(f'No points to look at (POINT_SET = {POINT_SET!r}).')
margin = 0.02
BBOX = (pts['lon'].min() - margin, pts['lat'].min() - margin, pts['lon'].max() + margin, pts['lat'].max() + margin)
print(f'{len(pts)} points ({POINT_SET}); search box {tuple(round(v, 3) for v in BBOX)}.')

## 4. Find the Landsat scenes

NASA's catalogue lists the HLSL30 scenes. Each covers one 110 km MGRS tile, so several tiles cover the district on each date. The table goes to `hls_l30_scenes_…csv`, with each scene's cloud cover and the number of points inside it.

In [ ]:
granules = earthaccess.search_data(short_name='HLSL30', version='2.0', bounding_box=BBOX,
                                   temporal=(HLS_START, HLS_END))
scenes = [scene_info(g) for g in granules]
point_geoms = {pid: Point(lon, lat) for pid, lon, lat in pts[['point_id', 'lon', 'lat']].itertuples(index=False)}
for s in scenes:
    # Points inside the scene's outline (all points if CMR gave no outline); the windows decide in the end.
    s['candidates'] = ([pid for pid, p in point_geoms.items() if s['outline'].contains(p)] if s['outline'] is not None
                       else list(point_geoms))
    s['used'] = (s['cloud_cover_pct'] <= MAX_SCENE_CLOUD) and bool(s['candidates'])
scene_table = pd.DataFrame([{k: s[k] for k in ('scene', 'tile', 'date', 'cloud_cover_pct', 'used')}
                            | {'points_inside': len(s['candidates'])} for s in scenes])
if scene_table.empty:
    raise RuntimeError(f'NASA lists no HLSL30 scenes over the points from {HLS_START} to {HLS_END}.')
scene_table = scene_table.sort_values(['date', 'tile']).reset_index(drop=True)
scene_table.assign(date=scene_table['date'].dt.strftime('%Y-%m-%d')).to_csv(
    PROCESSED / f'hls_l30_scenes_{SUFFIX}.csv', index=False)
by_date = scene_table.groupby(scene_table['date'].dt.strftime('%d %b'), sort=False).agg(
    tiles=('tile', 'count'), cloud_min=('cloud_cover_pct', 'min'), cloud_max=('cloud_cover_pct', 'max'),
    used=('used', 'sum'))
print(f'{len(scene_table)} scenes on {len(by_date)} dates; {scene_table["used"].sum()} to read '
      f'(cloud cover up to {MAX_SCENE_CLOUD}% and points inside).')
display(by_date)

## 5. Read small windows around the points

Scenes already in the cache are not read again. With `WORKERS = 6`, six scenes are read at a time. A scene that fails three times is skipped with a warning; run the cell again to retry it.

In [ ]:
todo, cached = [], 0
for s in scenes:
    if not s['used']:
        continue
    hit = load_scene(s['scene'])
    if hit is not None and set(s['candidates']) <= hit[1]:
        cached += 1
    else:
        todo.append(s)
print(f'{cached} scenes cached, {len(todo)} to read.')

files_opened, failed = 0, []
if todo:
    started = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = {pool.submit(read_scene, s, pts[pts['point_id'].isin(s['candidates'])]): s for s in todo}
        for i, future in enumerate(as_completed(futures), 1):
            s = futures[future]
            try:
                found, opened = future.result()
            except Exception as err:          # one scene failing must not stop the others
                failed.append(s['scene'])
                print(f'  WARNING: {s["scene"]} failed ({type(err).__name__}: {str(err)[:120]}); run again to retry.')
                continue
            files_opened += opened
            save_scene(s['scene'], found, s['candidates'])
            if i % 5 == 0 or i == len(todo):
                print(f'  {i} of {len(todo)} scenes read ({time.time() - started:.0f} s)')
print(f'Opened {files_opened} image files' + (f'; {len(failed)} scenes failed.' if failed else '.'))

## 6. Strips and the list for relabelling

Each point gets one view per date. Where two tiles overlap, the first tile read is kept. Dates with cloud, cloud shadow or a cloud edge over the point's block are left off the strip but listed in the table. *Haze* marks dates on which HLS flags high aerosol, often smoke from burning.

In [ ]:
views = {pid: {} for pid in pts['point_id']}       # point -> date -> view
not_clear = {pid: set() for pid in pts['point_id']}
for s in sorted(scenes, key=lambda s: (s['date'], s['tile'])):
    hit = load_scene(s['scene']) if s['used'] else None
    if hit is None:
        continue
    for pid, v in hit[0].items():
        if pid not in views or s['date'] in views[pid]:
            continue
        if v['clear'] and all(b in v for b in BANDS.values()):
            views[pid][s['date']] = v
        else:
            not_clear[pid].add(s['date'])

CHIPS.mkdir(parents=True, exist_ok=True)
gap_start, gap_end = (pd.Timestamp(d) for d in SENTINEL2_GAP)
rows = []
for pid, note in pts[['point_id', 'notes']].itertuples(index=False):
    dated = sorted(views[pid].items())
    landsat_strip(pid, dated, CHIPS / f'{pid}_landsat.png')
    clear_days = [d for d, _ in dated]
    rows.append({'point_id': pid, 'landsat_file': f'data/raw/validation_chips/{pid}_landsat.png', 'notes': note,
                 'clear_landsat_dates': len(clear_days),
                 'clear_in_sentinel2_gap': sum(gap_start <= d <= gap_end for d in clear_days),
                 'clear_dates': ';'.join(f'{d:%Y-%m-%d}' for d in clear_days),
                 'hazy_dates': ';'.join(f'{d:%Y-%m-%d}' for d, v in dated if v['hazy']),
                 'not_clear_dates': ';'.join(f'{d:%Y-%m-%d}' for d in sorted(not_clear[pid] - set(clear_days)))})
listing = pd.DataFrame(rows)
listing.to_csv(PROCESSED / f'validation_landsat_views_{SUFFIX}.csv', index=False)
print(f'Drew {len(listing)} strips. {(listing["clear_landsat_dates"] > 0).sum()} of {len(listing)} points have at '
      f'least one clear Landsat date; {(listing["clear_in_sentinel2_gap"] > 0).sum()} have one from '
      f'{gap_start:%d %b} to {gap_end:%d %b}, when Sentinel-2 had no clear view of most gap points.')
display(listing[['point_id', 'clear_landsat_dates', 'clear_in_sentinel2_gap', 'clear_dates', 'notes']])

register_source(
    source_id='hls_l30_v2_lpdaac_earthdata',
    dataset_or_product=('HLS Landsat Operational Land Imager Surface Reflectance and TOA Brightness Daily Global '
                        '30m v2.0 (HLSL30): bands B02, B03, B04, B05, B07 and Fmask'),
    provider='NASA Land Processes DAAC (LP DAAC), through NASA Earthdata; Masek et al. (2021)',
    url='https://doi.org/10.5067/HLS/HLSL30.002', date_accessed=ACCESS_DATE,
    geography=f'Windows of {PANEL_PX * 30} m around {len(pts)} validation points ({POINT_SET}) in Sangrur, Punjab',
    time_range=f'{HLS_START} to {HLS_END}',
    resolution='30 m; Landsat 8 and 9, each every 16 days',
    licence_or_terms='NASA Earth science data: free and open; cite the dataset DOI',
    preprocessing=(f'Scenes with cloud cover above {MAX_SCENE_CLOUD}% skipped; a date is shown only where the Fmask '
                   "layer marks most of the point's 60 m block free of cloud, cloud shadow and cloud edge"),
    known_limitations=('30 m pixels blur small fields and edges; Fmask may not flag smoke haze as cloud (high '
                       'aerosol is flagged separately); used only to interpret labels by eye'),
    used_in=NOTEBOOK)
print('Source register updated.')

### How to relabel

Open `data/processed/validation_landsat_views_sangrur_2025.csv`. For each point, look at `P001_landsat.png` next to its Sentinel-2 strip `P001_dates.png`. The Landsat pixels are 30 m, three times coarser than the Sentinel-2 dates, so judge the field around the square, not single pixels. Then edit the label sheet, `validation_labels_sangrur_2025.csv`:
- **`unsure` → `yes`:** the field is black or near-black on a Landsat date, most often 27 or 28 Oct, inside the Sentinel-2 gap. Add `Landsat 27 Oct` (the date) to the notes.
- **`unsure` → `no`:** only if no gap is left. A clear view, Landsat or Sentinel-2, shows straw at most 3 days before the first plain-soil view, and no date looks burnt. Add `Landsat 27 Oct: no burn` to the notes.
- **Otherwise** leave it `unsure` and keep its reason. A `dull` or `edge` point rarely changes: 30 m pixels don't help there.

Then close the sheet, open Notebook 02, set `RUN_FULL = True` and click Run All. Section 11 recomputes the accuracy, the burnt area from the labels, and the unsure bounds.

### Caveats
- **Coarser pixels.** At 30 m, a 1 ha field is about 3 × 3 pixels, so edges and small burns blur.
- **Haze.** Fmask flags cloud, but smoke usually shows only as a high-aerosol flag. Dates marked *(haze)* can look darker than they are.
- **Fewer dates.** Each Landsat satellite repeats every 16 days; together, and with two orbital paths over the district, a point gets a view every few days at best, often less.